# 09 — Phase 0a data-loader smoke test

**Purpose**: visual sanity check that the multi-target planner reads the *exact* real-data shapefiles used by 07.

**Owner**: FPO-5 / Phase 0a / 0a.1 (FPO engineer).

What this notebook does:
1. Calls `multi_target_planner.verify_data_sources()` to assert every expected file is present in `C:\nasa-ffp-nurture\test_data\`.
2. Loads all 7 input layers (3-TPV contour file, restricted airspace, ATC zone, dropsonde zones, satellite track, gatepoints, BASE) into a single local km frame.
3. Plots them all on one figure for Master to eyeball.

**Loud-fail policy**: if any file is missing the cell raises immediately. No fallback to synthetic data (LOCK / FPO-5 amendment 2026-06-09).

In [ ]:
%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.lines import Line2D
from matplotlib.patches import Patch

from multi_target_planner import (
    DEFAULT_DATA_DIR,
    TPV_FILENAME_PHASE0A,
    verify_data_sources,
    load_tpvs,
    load_restricted,
    load_atc,
    load_dropsonde,
    load_satellite_track,
    load_gatepoints,
    base_km,
)

# Step 1: file-presence check (loud fail on missing).
report = verify_data_sources(DEFAULT_DATA_DIR)
missing = [(role, path) for role, (path, ok) in report.items() if not ok]
if missing:
    raise FileNotFoundError(
        'Missing data files (Phase 0a forbids synthetic substitution):\n  '
        + '\n  '.join(f'{r}: {p}' for r, p in missing)
    )
print('All 7 data sources present in', DEFAULT_DATA_DIR)

# Step 2: load all layers in a single local km frame centred on the TPV union.
tpvs, frame = load_tpvs(DEFAULT_DATA_DIR, TPV_FILENAME_PHASE0A)
restricted = load_restricted(DEFAULT_DATA_DIR, frame)
atc        = load_atc(DEFAULT_DATA_DIR, frame)
dropsonde  = load_dropsonde(DEFAULT_DATA_DIR, frame)
satellite  = load_satellite_track(DEFAULT_DATA_DIR, frame)
gatepoints = load_gatepoints(DEFAULT_DATA_DIR, frame)
base       = base_km(frame)

print(f'TPVs loaded     : {len(tpvs)}')
for t in tpvs:
    cx, cy = t.polygon_km.centroid.x, t.polygon_km.centroid.y
    print(f'  {t.label}: {len(t.vertices_km)} vertices, centroid=({cx:.0f}, {cy:.0f}) km, area={t.polygon_km.area:.0f} km^2')
print(f'Restricted polys: {len(restricted)}  (clipped to 2000 km of TPV centroid)')
print(f'ATC polys       : {len(atc)}         (Gander FIR)')
print(f'Dropsonde polys : {len(dropsonde)}   (clipped to 2000 km)')
print(f'Satellite track : {satellite.shape[0]} vertices, span {np.ptp(satellite[:,0]):.0f}x{np.ptp(satellite[:,1]):.0f} km')
print(f'Gatepoints      : {gatepoints.shape[0]}')
print(f'BASE (GOOSE BAY): ({base[0]:.0f}, {base[1]:.0f}) km from frame origin')
print(f'Frame origin    : lon_c={frame.lon_c:.3f}, lat_c={frame.lat_c:.3f}')

In [ ]:
# Step 3: single-figure overlay of every layer.
fig, ax = plt.subplots(figsize=(12, 11))

# Restricted (red), ATC (yellow), dropsonde (green): draw bottom-up so TPVs sit on top.
for poly in dropsonde:
    xs, ys = poly.exterior.xy
    ax.fill(xs, ys, color='#22bb55', alpha=0.18, zorder=2)
for poly in atc:
    xs, ys = poly.exterior.xy
    ax.fill(xs, ys, color='#ffe066', alpha=0.40, zorder=3)
    ax.plot(xs, ys, color='#b8860b', lw=1.3, zorder=3.5)
for poly in restricted:
    xs, ys = poly.exterior.xy
    ax.fill(xs, ys, color='#ff6666', alpha=0.45, zorder=4)
    ax.plot(xs, ys, color='#cc0000', lw=1.3, zorder=4.5)

# Satellite track: dashed purple line (global track; xlim below clips to mission area).
ax.plot(satellite[:, 0], satellite[:, 1], '--', color='#8800bb', lw=1.5,
        alpha=0.7, zorder=5, label='Satellite overpass (EarthCARE 2016-05-16)')

# TPV polygons - distinct colours so multi-TPV is obvious.
tpv_colors = ['#1f77b4', '#d62728', '#2ca02c']
for tpv, col in zip(tpvs, tpv_colors):
    verts = tpv.vertices_km
    closed = np.vstack([verts, verts[0]])
    ax.fill(verts[:, 0], verts[:, 1], color=col, alpha=0.25, zorder=6)
    ax.plot(closed[:, 0], closed[:, 1], color=col, lw=1.6, zorder=6.5,
            label=f'{tpv.label} polygon')
    cx, cy = tpv.polygon_km.centroid.x, tpv.polygon_km.centroid.y
    ax.annotate(tpv.label, (cx, cy), color=col, fontweight='bold',
                ha='center', va='center', fontsize=10, zorder=10)

# Gatepoints - orange diamonds, numbered.
for i, (gx, gy) in enumerate(gatepoints):
    ax.plot(gx, gy, marker='D', color='#ff7f0e', ms=11, zorder=8)
    ax.annotate(f'GP{i}', (gx, gy), textcoords='offset points', xytext=(8, 4),
                fontsize=9, fontweight='bold', color='#cc5500', zorder=10)

# BASE - black star.
ax.plot(base[0], base[1], marker='*', color='black', ms=20, zorder=9)
ax.annotate('BASE (GOOSE BAY)', (base[0], base[1]),
            textcoords='offset points', xytext=(10, 6),
            fontsize=10, fontweight='bold', zorder=10)

# Legend handled via proxy artists so polygon layers are labelled cleanly.
legend_handles = [
    Line2D([0], [0], color=tpv_colors[0], lw=2, label='TPV-1'),
    Line2D([0], [0], color=tpv_colors[1], lw=2, label='TPV-2'),
    Line2D([0], [0], color=tpv_colors[2], lw=2, label='TPV-3'),
    Patch(facecolor='#ff6666', alpha=0.45, label='Restricted airspace (no-fly)'),
    Patch(facecolor='#ffe066', alpha=0.40, label='ATC zone (Gander FIR, x1.35)'),
    Patch(facecolor='#22bb55', alpha=0.25, label='Dropsonde-deployment zones'),
    Line2D([0], [0], color='#8800bb', lw=1.5, ls='--',
           label='Satellite track (EarthCARE 0516)'),
    Line2D([0], [0], marker='*', color='black', ms=12, ls='none', label='BASE (GOOSE BAY)'),
    Line2D([0], [0], marker='D', color='#ff7f0e', ms=9, ls='none', label='Mandatory gatepoint'),
]
ax.legend(handles=legend_handles, fontsize=9, loc='upper right', framealpha=0.92)

# Mission bounding box: TPV union extent + BASE + gatepoints + 500 km padding.
# Satellite track is global and would dominate the view otherwise - we clip the
# viewport to the mission area so TPVs are readable.
all_x = np.concatenate([t.vertices_km[:, 0] for t in tpvs] + [[base[0]], gatepoints[:, 0]])
all_y = np.concatenate([t.vertices_km[:, 1] for t in tpvs] + [[base[1]], gatepoints[:, 1]])
pad = 500.0
ax.set_xlim(all_x.min() - pad, all_x.max() + pad)
ax.set_ylim(all_y.min() - pad, all_y.max() + pad)
ax.set_aspect('equal')
ax.set_xlabel('Easting (km from TPV-union centroid)', fontsize=11)
ax.set_ylabel('Northing (km from TPV-union centroid)', fontsize=11)
ax.set_title(
    'FPO-5 / Phase 0a - input data smoke check\n'
    f'TPV source: {TPV_FILENAME_PHASE0A}  (3 TPVs)   '
    f'All 7 real-data layers loaded from {DEFAULT_DATA_DIR}',
    fontsize=11,
)
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('09_phase0a_data_smoke.png', dpi=150, bbox_inches='tight')
plt.show()
print('Saved 09_phase0a_data_smoke.png')